**Bronze Layer ingestion**

In [0]:
# Read the original .txt files from Raw Volume into Spark cluster's memory as DataFrames

RAW_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/raw_volume/"

DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/"

gtfs_files = ["agency.txt",
              "routes.txt",
              "trips.txt",
              "calendar.txt",
              "stops.txt",
              "stop_times.txt"
]

raw_gtfs_dataframes = {}

for file in gtfs_files:
  df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(RAW_VOLUME_PATH + file)
  )

  df_name = "df_" + file.replace(".txt", "")
  # save it into the raw_gtfs_dataframes dictionary, using the clean name like(df_agency) as the key
  raw_gtfs_dataframes[df_name] = df
  print(f"loaded {file} as {df_name} ")

In [0]:
# Write DataFrames to the Bronze Delta Layer

for df_name, df in raw_gtfs_dataframes.items():

    table_name = df_name.replace("df_", "raw_")
    output_path = f"{DELTA_VOLUME_PATH}bronze/{table_name}/"


    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_path)
    )

    print(f"written {table_name} to {output_path}")

print("\nIngestion complete.")


In [0]:
CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "bronze"


print("\nBronze Layer Ingestion and Managed Registration Complete.")
# Managed Table Creation and Registration
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")

# Loop through all DataFrames and write them as Managed Tables
for df_name, df in raw_gtfs_dataframes.items():
    table_name = df_name.replace("df_", "raw_")
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"

    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )
    print(f"Successfully created and registered Managed Table: {full_table_name}")

print("\nBronze Layer Ingestion and Managed Registration Complete.")